# Notebook 5 — Merged-Adapter Ablation (decomposition hypothesis test)

Compares the **sequential specialist ensemble** (NB2 results) against a **single generalist adapter** fine-tuned on all three category datasets merged, under the identical corrected protocol.

**Workflow:**
- **Step A (once):** set `PUSH_MERGED_DATASET = True` in section 6, run sections 1–6 to build and push `fyp-slm-merged`, then set it back to `False`.
- **Step B (outside this notebook):** fine-tune ONE adapter on the merged dataset using your existing fine-tuning notebook with **identical hyperparameters** to the specialists (same LoRA rank, alpha, target modules, epochs, learning rate, seed). Push it to the Hub and set `MERGED_ADAPTER_REPO` in section 4 to its repo id. Record the training time — it goes in the comparison table.
- **Step C:** run this notebook top to bottom for the evaluation and comparison.

Note for the thesis: the merged adapter sees ~3x each specialist's training data under the same protocol — same recipe, different data budget; state this framing caveat in §7.6.

Outputs: merged-adapter metrics (threshold + score-based) on the same combined test set as NB2, per-category recall, ensemble-vs-merged comparison table, and `per_sample_predictions_merged.csv` for NB4's McNemar test.

## 1. Install

In [1]:
# %%capture
# !pip install --no-cache-dir -U transformers peft datasets scikit-learn pandas accelerate huggingface_hub bitsandbytes matplotlib

## 2. Imports

In [2]:
import os
import re
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from huggingface_hub import login
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             roc_curve, roc_auc_score, average_precision_score)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True if torch.cuda.is_available() else False

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.2f} GB")

CUDA available: True
GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU | 6.00 GB


## 3. Hugging Face login

In [3]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle secrets.")
except Exception:
    pass

if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        if HF_TOKEN:
            print("Loaded HF_TOKEN from Colab secrets.")
    except Exception:
        pass

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

## 4. Configuration

In [4]:
HF_USERNAME = "hirushafernando"

CATEGORIES = [
    "role-and-instruction-violation",    # SLM-A
    "privilege-escalation",              # SLM-B
    "obfuscation-and-evasion-patterns",  # SLM-C
]

DATASET_REPOS = {
    "role-and-instruction-violation": f"{HF_USERNAME}/fyp-slm-a",
    "privilege-escalation": f"{HF_USERNAME}/fyp-slm-b",
    "obfuscation-and-evasion-patterns": f"{HF_USERNAME}/fyp-slm-c",
}

MERGED_DATASET_REPO = f"{HF_USERNAME}/fyp-slm-merged"
MERGED_ADAPTER_REPO = f"{HF_USERNAME}/fyp-gemma3-1b-slm-merged-qlora"  # set to your training run's hub_model_id

BASE_MODEL = "google/gemma-3-1b-it"
EVAL_SPLIT = "test"

# Optional: upload NB2's system_evaluation_results.json to enable the comparison table
SYSTEM_RESULTS_JSON = "system_evaluation_results.json"

BATCH_SIZE = 8            # Kaggle T4: 8-16 | RTX 3060 6 GB: 4
MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 6
LOAD_IN_4BIT = True
TARGET_FPRS = [0.001, 0.01, 0.05]
QUICK_TEST = False
QUICK_N_PER_GROUP = 250
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 5. Data helpers (identical to NB1-3: corrected prompt, leakage-safe)

In [5]:
MODEL_TURN_RE = re.compile(r"<start_of_turn>model\s*")
ANSWER_TAIL_RE = re.compile(r"^\s*(BENIGN|INJECTION)\s*<end_of_turn>\s*$")

def strip_bos(t):
    return t[len("<bos>"):] if t.startswith("<bos>") else t

def canon(t):
    return re.sub(r"\s+", " ", t.strip().lower())

def make_gen_prompt(formatted_text):
    matches = list(MODEL_TURN_RE.finditer(formatted_text))
    if not matches:
        return None
    m = matches[-1]  # LAST model turn: attacks may embed fake chat turns in user content
    if ANSWER_TAIL_RE.match(formatted_text[m.end():]) is None:
        return None
    return formatted_text[:m.end()]

def extract_raw_prompt(formatted_text):
    a = formatted_text.find("User Prompt:")
    b = formatted_text.rfind("Respond with exactly one word")
    if a == -1 or b == -1 or b <= a:
        return None
    return formatted_text[a + len("User Prompt:"):b]

def load_split(cat, split=None):
    d = load_dataset(DATASET_REPOS[cat], split=split or EVAL_SPLIT, token=HF_TOKEN).to_pandas()
    d["formatted_text"] = d["formatted_text"].map(strip_bos)
    d["label"] = d["label"].astype(int)
    d["gen_prompt"] = d["formatted_text"].map(make_gen_prompt)
    d["raw_prompt"] = d["formatted_text"].map(extract_raw_prompt)
    d["source"] = cat
    d["category"] = d["label"].map(lambda y: cat if y == 1 else "benign")
    n_bad = int(d["gen_prompt"].isna().sum())
    d = d.dropna(subset=["gen_prompt", "raw_prompt"]).reset_index(drop=True)
    print(f"{cat}: {len(d):,} rows | benign={int((d.label==0).sum()):,} | "
          f"injection={int((d.label==1).sum()):,} | unparseable dropped={n_bad}")
    return d

## 6. Step A — build and push the merged training dataset (run ONCE)

Concatenates each split across the three datasets **as-is** (every sample keeps its own category instruction inside `formatted_text`), deduplicates benign rows by raw prompt within each split, shuffles, and pushes. Training on mixed instructions matches how the ensemble's evaluation prompts are constructed, so the comparison is fair.

In [7]:
PUSH_MERGED_DATASET = True   # set True once, run, then set back to False

if PUSH_MERGED_DATASET:
    from datasets import Dataset, DatasetDict

    merged_splits = {}
    for split in ["train", "validation", "test"]:
        fr = []
        for cat in CATEGORIES:
            df = load_dataset(DATASET_REPOS[cat], split=split, token=HF_TOKEN).to_pandas()
            df["label"] = df["label"].astype(int)
            fr.append(df)
        m = pd.concat(fr, ignore_index=True)
        n0 = len(m)
        m["_canon"] = m["formatted_text"].map(lambda t: canon(extract_raw_prompt(strip_bos(t)) or t))
        m = m.drop_duplicates(subset=["label", "_canon"]).drop(columns="_canon")
        m = m.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        merged_splits[split] = Dataset.from_pandas(m, preserve_index=False)
        print(f"{split}: {n0:,} -> {len(m):,} after dedup on raw prompt")

    DatasetDict(merged_splits).push_to_hub(MERGED_DATASET_REPO, token=HF_TOKEN, private=True)
    print("\nPushed:", MERGED_DATASET_REPO)
    print("Now fine-tune ONE adapter on this dataset with your existing fine-tuning notebook "
          "(identical hyperparameters), push it, set MERGED_ADAPTER_REPO, and re-run this notebook.")
else:
    print("Skipped (PUSH_MERGED_DATASET = False)")

train: 184,792 -> 108,798 after dedup on raw prompt
validation: 23,100 -> 21,489 after dedup on raw prompt
test: 23,100 -> 21,492 after dedup on raw prompt


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Pushed: hirushafernando/fyp-slm-merged
Now fine-tune ONE adapter on this dataset with your existing fine-tuning notebook (identical hyperparameters), push it, set MERGED_ADAPTER_REPO, and re-run this notebook.


## 7. Build the combined test set (identical to NB2)

In [ ]:
frames = [load_split(cat)[["gen_prompt", "raw_prompt", "label", "source", "category"]]
          for cat in CATEGORIES]
combined = pd.concat(frames, ignore_index=True)
n_before = len(combined)
combined["_canon"] = combined["raw_prompt"].map(canon)
combined = combined.drop_duplicates(subset=["label", "_canon"]).drop(columns="_canon").reset_index(drop=True)
print(f"\nCombined: {n_before:,} -> {len(combined):,} after dedup on raw user prompt")
print(combined.groupby("category").size().to_string())

if QUICK_TEST:
    combined = (combined.groupby("category", group_keys=False)
                .apply(lambda g: g.sample(min(QUICK_N_PER_GROUP, len(g)), random_state=SEED))
                .reset_index(drop=True))
    print(f"\nQUICK_TEST subsample: {len(combined):,} rows")

gen_prompts = combined["gen_prompt"].tolist()
y_true = combined["label"].to_numpy()
true_cat = combined["category"].to_numpy()
benign_mask = y_true == 0
N = len(gen_prompts)

## 8. Load 4-bit backbone + the MERGED adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_cfg = None
if LOAD_IN_4BIT:
    quant_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16,
                                   bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_cfg, dtype=torch.float16,
    device_map="auto", token=HF_TOKEN)
model = PeftModel.from_pretrained(model, MERGED_ADAPTER_REPO, adapter_name="merged", token=HF_TOKEN)
model.eval()
print("Loaded merged adapter:", MERGED_ADAPTER_REPO)

## 9. Inference (verdict + score in one pass)

In [ ]:
def first_token_ids(word):
    ids = set()
    for w in (word, " " + word, "\n" + word):
        toks = tokenizer.encode(w, add_special_tokens=False)
        for t in toks:
            if tokenizer.decode([t]).strip():   # skip pure-whitespace tokens
                ids.add(t)
                break
    return sorted(ids)

INJ_IDS = first_token_ids("INJECTION")
BEN_IDS = sorted(set(first_token_ids("BENIGN") + first_token_ids("SAFE")))
assert not set(INJ_IDS) & set(BEN_IDS), "Candidate token sets overlap"

unparsed_counts = {}

@torch.inference_mode()
def infer_batch(prompts, run_key):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                       truncation=True, max_length=MAX_INPUT_TOKENS).to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        output_scores=True, return_dict_in_generate=True)
    decoded = tokenizer.batch_decode(out.sequences[:, inputs["input_ids"].shape[1]:],
                                     skip_special_tokens=True)
    lp = torch.log_softmax(out.scores[0].float(), dim=-1)
    scores = (torch.logsumexp(lp[:, INJ_IDS], dim=-1)
              - torch.logsumexp(lp[:, BEN_IDS], dim=-1)).cpu().numpy()
    preds = []
    for d in decoded:
        d = d.strip().upper()
        if "INJECTION" in d:
            preds.append(1)
        elif "BENIGN" in d or "SAFE" in d:
            preds.append(0)
        else:
            preds.append(1)
            unparsed_counts[run_key] = unparsed_counts.get(run_key, 0) + 1
    return preds, scores


def run_inference(run_key, prompts, progress_every=50):
    Np = len(prompts)
    all_preds, all_scores = [], []
    t_start = time.perf_counter()
    for b, s in enumerate(range(0, Np, BATCH_SIZE)):
        p, sc = infer_batch(prompts[s:s + BATCH_SIZE], run_key)
        all_preds.extend(p); all_scores.extend(sc.tolist())
        if b % progress_every == 0:
            done = min(s + BATCH_SIZE, Np)
            el = time.perf_counter() - t_start
            print(f"[{run_key}] {done}/{Np} | elapsed {el/60:.1f} min | ETA {el/done*(Np-done)/60:.1f} min")
    return np.array(all_preds), np.array(all_scores)

## 10. Metric helpers

In [ ]:
def binary_metrics(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="binary",
                                                       pos_label=1, zero_division=0)
    return {"accuracy": float(accuracy_score(y, p)),
            "precision_injection": float(prec), "recall_injection": float(rec),
            "f1_injection": float(f1),
            "fpr": float(fp / (fp + tn)) if (fp + tn) else 0.0,
            "fnr": float(fn / (fn + tp)) if (fn + tp) else 0.0,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def tpr_at_fpr(y, s, target):
    fpr, tpr, _ = roc_curve(y, s)
    idx = np.searchsorted(fpr, target, side="right") - 1
    return float(tpr[max(idx, 0)])

def score_metrics(y, s):
    out = {"auroc": float(roc_auc_score(y, s)),
           "pr_auc": float(average_precision_score(y, s))}
    for t in TARGET_FPRS:
        out[f"tpr_at_fpr_{t}"] = tpr_at_fpr(y, s, t)
    return out

## 11. Evaluate the merged adapter on the combined set (native prompts, as trained)

In [ ]:
merged_pred, merged_score = run_inference("merged", gen_prompts)

merged_metrics = binary_metrics(y_true, merged_pred)
merged_metrics.update(score_metrics(y_true, merged_score))
merged_metrics["per_category_recall"] = {c: float(merged_pred[true_cat == c].mean())
                                         for c in CATEGORIES}
merged_metrics["unparsed_fail_closed"] = unparsed_counts.get("merged", 0)
merged_metrics["eval_rows"] = int(N)
print(json.dumps(merged_metrics, indent=2))

cm = confusion_matrix(y_true, merged_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(3.4, 3))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["BENIGN", "INJECTION"]); ax.set_yticks([0, 1], ["BENIGN", "INJECTION"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Merged single adapter")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "cm_merged.png"), dpi=200); plt.show()

## 12. Comparison: sequential ensemble vs. merged single adapter

Loads NB2's `system_evaluation_results.json` if available (upload it as an input file or place it next to this notebook). The McNemar significance test runs in NB4 using the per-sample CSV saved below.

In [ ]:
if os.path.exists(SYSTEM_RESULTS_JSON):
    with open(SYSTEM_RESULTS_JSON) as f:
        system = json.load(f)["system_or_gate"]
    KEYS = ["accuracy", "precision_injection", "recall_injection", "f1_injection", "fpr", "fnr"]
    cmp = pd.DataFrame({"sequential_ensemble_OR": {k: system[k] for k in KEYS},
                        "merged_single_adapter": {k: merged_metrics[k] for k in KEYS}})
    cmp.loc["adapters_at_runtime"] = [3, 1]
    print(cmp.round(4).to_string())
    cmp.to_csv(os.path.join(OUTPUT_DIR, "ablation_ensemble_vs_merged.csv"))
    print("\nPer-category recall (merged):",
          json.dumps(merged_metrics["per_category_recall"], indent=2))
else:
    print(f"'{SYSTEM_RESULTS_JSON}' not found - upload NB2's results file to enable the table.")
    print("Merged metrics above are still saved; the comparison can be assembled manually.")

## 13. Save results (incl. per-sample predictions for NB4)

In [ ]:
with open(os.path.join(OUTPUT_DIR, "merged_ablation_results.json"), "w") as f:
    json.dump({"protocol": "corrected-native-prompt", "adapter": MERGED_ADAPTER_REPO,
               "eval_split": EVAL_SPLIT, "quick_test": QUICK_TEST, "seed": SEED,
               "merged_metrics": merged_metrics}, f, indent=2)

dump = combined[["label", "source", "category"]].copy()
dump["pred_merged"] = merged_pred
dump["score_merged"] = merged_score
dump.to_csv(os.path.join(OUTPUT_DIR, "per_sample_predictions_merged.csv"), index=False)
print("Saved:", sorted(os.listdir(OUTPUT_DIR)))
print("\nNext: feed per_sample_predictions_merged.csv to NB4 for the McNemar ablation test.")

---
**Thesis use (§7.6):** `ablation_ensemble_vs_merged.csv` is the decomposition hypothesis table; combine with NB4's system-vs-merged McNemar p-value. Given the leakage matrix already showed partial generalisation across categories, interpret either outcome honestly: if the merged adapter matches the ensemble on detection, the decomposition's remaining value is category attribution and modular extensibility (new attack category = train one adapter, no retraining of the others) — argue it on those grounds, not raw F1.